# 第二篇、NumPy快速入门：数组是什么，为什么快?

> 这是「数据分析从入门到精通」系列的第 2 篇。上一篇我们画了整个学习路线图，从这篇开始，正式动手。

## 先回答一个问题：为什么不直接用 Python 列表？

学 NumPy 之前，很多人会有一个疑问：

> Python 已经有列表（list）了，可以存数据、可以循环、可以切片——为什么还要专门学 NumPy？

来看一个实验，一眼就懂：

In [1]:

import numpy as np
import time

# 准备一百万个数字
n = 1_000_000
py_list = list(range(n))
np_array = np.arange(n)

# Python 列表：每个元素乘以 2
start = time.time()
result = [x * 2 for x in py_list]
print(f"Python 列表耗时：{time.time() - start:.4f} 秒")

# NumPy 数组：每个元素乘以 2
start = time.time()
result = np_array * 2
print(f"NumPy 数组耗时：{time.time() - start:.4f} 秒")

Python 列表耗时：0.0457 秒
NumPy 数组耗时：0.0140 秒


## NumPy 为什么这么快？

说透了其实就三个原因：

**① 连续内存存储**

Python 列表里的每个元素，是一个独立的 Python 对象，分散存储在内存各处。CPU 读取时要到处"跑"，效率很低。

NumPy 数组把所有元素**紧密排列在一块连续的内存区域**，CPU 可以一次性批量读取，像流水线一样处理。

```
Python 列表（内存分散）：
[obj → 内存A] [obj → 内存B] [obj → 内存C] ...

NumPy 数组（内存连续）：
[1.0][2.0][3.0][4.0][5.0]... （紧密排列）
```

**② 类型统一**

Python 列表可以混装任意类型：`[1, "hello", True, 3.14]`。灵活是灵活，但每次操作都要判断类型，很慢。

NumPy 数组**所有元素类型相同**，比如都是 `float64`。不需要类型判断，运算直接执行。

**③ 向量化运算（底层用 C 实现）**

NumPy 的运算底层是用 **C 语言**写的，绕过了 Python 解释器的开销。`array * 2` 这一行代码，背后调用的是高度优化的 C 函数，直接在 CPU 层面批量处理。

> 简单记：NumPy = 连续内存 + 统一类型 + C 语言底层 → 快。

## ndarray：NumPy 的核心对象

NumPy 里最重要的东西只有一个——**ndarray**（n 维数组，n-dimensional array）。

所有操作都围绕它展开。

In [2]:
# 创建一个数组
import numpy as np

# 1. 从 Python 列表创建
a = np.array([1, 2, 3, 4, 5])
print(a)        # [1 2 3 4 5]
print(type(a))  # <class 'numpy.ndarray'>

[1 2 3 4 5]
<class 'numpy.ndarray'>


In [3]:
# 2. 等差数列（最常用）
np.arange(0, 10, 2)       # [0 2 4 6 8]

array([[0.33119292, 0.19206091, 0.44385746],
       [0.80081056, 0.885787  , 0.21166274],
       [0.08834806, 0.44288267, 0.90228306]])

In [4]:
# 3. 指定个数的等间隔
np.linspace(0, 1, 5)      # [0.   0.25  0.5  0.75  1. ]

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

In [5]:
# 4. 全零数组
np.zeros((3, 4))           # 3行4列，全是 0.0

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])

In [6]:
# 5. 全一数组
np.ones((2, 3))            # 2行3列，全是 1.0

array([[1., 1., 1.],
       [1., 1., 1.]])

In [7]:
# 6. 随机数组（0到1之间）
np.random.rand(3, 3)       # 3×3 随机矩阵

array([[0.77640561, 0.18027984, 0.74917336],
       [0.31213427, 0.25025266, 0.65047245],
       [0.70020402, 0.53945446, 0.96534335]])

## 数组的两个关键属性

In [8]:
a = np.array([[1, 2, 3],
              [4, 5, 6]])

print(a.shape)   # (2, 3) → 2行3列
print(a.dtype)   # int64  → 元素类型

(2, 3)
int64


## 向量化运算：告别 for 循环

NumPy 最重要的使用习惯：**用数组运算代替 for 循环**。

In [9]:
scores = np.array([72, 85, 91, 63, 78])

# ❌ 旧习惯：for 循环
result = []
for s in scores:
    result.append(s * 1.1)

# ✅ NumPy 方式：直接对数组操作
result = scores * 1.1
# [79.2 93.5 100.1 69.3 85.8]

不只是乘法，加减除、比较、函数——全都可以直接作用在整个数组上：

In [11]:
scores = np.array([72, 85, 91, 63, 78])

scores + 5          # 每个加5
scores

array([72, 85, 91, 63, 78])

In [12]:
scores > 80         # [False  True  True False False]

array([False,  True,  True, False, False])

In [13]:
np.sqrt(scores)     # 每个开根号

array([8.48528137, 9.21954446, 9.53939201, 7.93725393, 8.83176087])

In [14]:
np.mean(scores)     # 均值 → 77.8

np.float64(77.8)

## 实战：向量化 vs for 循环，差距有多大？
下面这个场景在数据分析里非常常见——对一列数据做标准化（减均值除以标准差）：

In [15]:
import numpy as np
import time

data = np.random.randn(1_000_000)  # 一百万个数

# 方法一：for 循环
start = time.time()
mean, std = data.mean(), data.std()
result1 = [(x - mean) / std for x in data]
print(f"for 循环耗时：{time.time() - start:.4f} 秒")

# 方法二：NumPy 向量化
start = time.time()
result2 = (data - data.mean()) / data.std()
print(f"NumPy 向量化耗时：{time.time() - start:.4f} 秒")

for 循环耗时：0.1101 秒
NumPy 向量化耗时：0.0101 秒


## 课后练习

试着自己跑一遍下面的代码，观察输出结果：

In [16]:
import numpy as np

# 1. 创建一个 1 到 20 的数组，只保留其中的偶数
a = np.arange(1, 21)
print(a[a % 2 == 0])

# 2. 创建一个 3×3 的随机整数矩阵（0-100之间）
b = np.random.randint(0, 100, size=(3, 3))
print(b)
print("每行最大值：", b.max(axis=1))
print("每列平均值：", b.mean(axis=0))

[ 2  4  6  8 10 12 14 16 18 20]
[[60 26 18]
 [41 61  5]
 [ 7 21  5]]
每行最大值： [60 61 21]
每列平均值： [36.         36.          9.33333333]
